In [4]:

import pandas as pd
from pathlib import Path

# =========================================================
# IMDb CROSS-SCALE COMPARISON
# =========================================================
# This notebook reproduces the same cross-scale comparison style
# used in the attached comparison_sfs notebook, but adapted to IMDb.
#
# IMPORTANT CHANGE:
# The corrected per-scale analysis now works at the granularity of:
#   (query_name, scale_label, run_phase)
# This is the correct granularity according to the paper.
#
# Because of that, this comparison notebook:
# 1) reads one analysis file per scale-factor result folder
# 2) filters by run_phase explicitly
# 3) keeps the same main output artifacts as the previous notebook
# 4) optionally also summarizes near-best preservation, because the
#    corrected individual analysis now computes it
#
# Expected input files (one per scale folder):
# - imdb_summary_hot_by_scale.csv
#   or
# - imdb_summary_hot.csv
#
# Main output artifacts:
# - imdb_cross_scale_schemalens_analysis_final.csv
# - imdb_cross_scale_summary_final.csv
# - imdb_cross_scale_secondary_winners_final.csv
# - imdb_cross_scale_control_winners_final.csv
#
# Optional extra artifact:
# - imdb_cross_scale_near_best_summary_final.csv
# =========================================================

# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------
base_results = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset imdb oficial/results"
)

# Adjust these paths to your actual IMDb result folders
paths = {
    "sf0_25": base_results / "sf_025" / "imdb_analysis_outputs" / "imdb_summary_hot_by_scale.csv",
    "sf0_50":  base_results / "sf_050"  / "imdb_analysis_outputs" / "imdb_summary_hot_by_scale.csv",
    "sf_1":    base_results / "sf_1"    / "imdb_analysis_outputs" / "imdb_summary_hot_by_scale.csv",
}

# If your exported files use the older name, change here:
fallback_filename = "imdb_summary_hot.csv"

selected_run_phase = "hot"

# ---------------------------------------------------------
# 2) Read all per-scale analysis files
# ---------------------------------------------------------
dfs = []

for scale, path in paths.items():
    if not path.exists():
        fallback_path = path.parent / fallback_filename
        if fallback_path.exists():
            path = fallback_path
        else:
            raise FileNotFoundError(
                f"Could not find analysis file for {scale}.\n"
                f"Tried:\n- {path}\n- {fallback_path}"
            )

    df = pd.read_csv(path)

    # Ensure scale label exists and is correct
    if "scale_label" not in df.columns:
        df["scale_label"] = scale

    # Force expected scale in case the file contains only one-scale results
    if df["scale_label"].nunique() == 1:
        df["scale_label"] = scale

    # Keep only selected run phase if present
    if "run_phase" in df.columns:
        df = df[df["run_phase"] == selected_run_phase].copy()
    else:
        df["run_phase"] = selected_run_phase

    dfs.append(df)

cross_scale_schemalens_df = pd.concat(dfs, ignore_index=True)

# ---------------------------------------------------------
# 3) Build the main summary table
# Same style as the attached notebook
# ---------------------------------------------------------
agg_dict = dict(
    n_queries=("query_name", "nunique"),
    avg_DSR=("DSR", "mean"),
    top1_preservation=("top1_preserved_by_activated", "mean"),
    mean_activated_regret=("activated_regret", "mean"),
    mean_primary_regret=("primary_regret", lambda x: x.dropna().mean()),
    primary_winners=("best_group", lambda x: int((x == "primary").sum())),
    secondary_winners=("best_group", lambda x: int((x == "secondary_affected").sum())),
    control_winners=("best_group", lambda x: int((x == "control").sum())),
)

# If the corrected file contains near-best preservation, include it
if "near_best_preserved_by_activated" in cross_scale_schemalens_df.columns:
    agg_dict["near_best_preservation"] = ("near_best_preserved_by_activated", "mean")

summary_df = (
    cross_scale_schemalens_df
    .groupby("scale_label")
    .agg(**agg_dict)
    .reset_index()
)

display(summary_df)

# ---------------------------------------------------------
# 4) Secondary winners table
# ---------------------------------------------------------
secondary_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    secondary_cols.insert(1, "run_phase")

secondary_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "secondary_affected"
][secondary_cols].sort_values(["scale_label", "official_id"])

display(secondary_df)

# ---------------------------------------------------------
# 5) Control winners table
# ---------------------------------------------------------
control_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
    "activated_regret",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    control_cols.insert(1, "run_phase")

control_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "control"
][control_cols].sort_values(["scale_label", "official_id"])

display(control_df)

# ---------------------------------------------------------
# 6) Optional extra: best-by-scale overview
# ---------------------------------------------------------
best_by_scale_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_group",
    "best_design_pattern",
    "best_p95_ms",
]

best_by_scale_df = cross_scale_schemalens_df[best_by_scale_cols].sort_values(
    ["scale_label", "official_id"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 7) Save outputs
# ---------------------------------------------------------
out_dir = base_results / "cross_scale_analysis_final"
out_dir.mkdir(parents=True, exist_ok=True)

cross_scale_schemalens_df.to_csv(out_dir / "imdb_cross_scale_schemalens_analysis_final.csv", index=False)
summary_df.to_csv(out_dir / "imdb_cross_scale_summary_final.csv", index=False)
secondary_df.to_csv(out_dir / "imdb_cross_scale_secondary_winners_final.csv", index=False)
control_df.to_csv(out_dir / "imdb_cross_scale_control_winners_final.csv", index=False)
best_by_scale_df.to_csv(out_dir / "imdb_cross_scale_best_by_scale_final.csv", index=False)

# Optional extra near-best table
if "near_best_preserved_by_activated" in cross_scale_schemalens_df.columns:
    near_best_summary_df = (
        cross_scale_schemalens_df
        .groupby("scale_label")
        .agg(
            near_best_preservation=("near_best_preserved_by_activated", "mean"),
            mean_activated_regret=("activated_regret", "mean"),
        )
        .reset_index()
    )
    near_best_summary_df.to_csv(out_dir / "imdb_cross_scale_near_best_summary_final.csv", index=False)
    display(near_best_summary_df)

print("Saved to:", out_dir)


,scale_label,n_queries,avg_DSR,top1_preservation,mean_activated_regret,mean_primary_regret,primary_winners,secondary_winners,control_winners,near_best_preservation
0,sf0_25,10,0.222222,0.6,0.062674,1.389554,3,3,4,0.8
1,sf0_50,10,0.222222,0.6,0.049949,3.845498,3,3,4,0.7
2,sf_1,10,0.222222,0.7,0.003540,3.872221,3,4,3,1.0


,scale_label,run_phase,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
1,sf0_25,hot,QG10,QG10_AdvancedSearchWatchItems,G9,containment_family,115.312090,G3,117.935234,0.022748
3,sf0_25,hot,QG3,QG3_RecommendationByGenreAndSubtype,G8,containment_family,5.637626,G2,6.176213,0.095534
9,sf0_25,hot,QG9,QG9_TopRatedSeriesByGenre,G7,containment_family,3.006610,G0,40.197583,12.369738
11,sf0_50,hot,QG10,QG10_AdvancedSearchWatchItems,G7,containment_family,171.641413,G3,174.507078,0.016696
13,sf0_50,hot,QG3,QG3_RecommendationByGenreAndSubtype,G8,containment_family,7.588797,G0,7.950161,0.047618
19,sf0_50,hot,QG9,QG9_TopRatedSeriesByGenre,G7,containment_family,2.054243,G2,78.136428,37.036610
21,sf_1,hot,QG10,QG10_AdvancedSearchWatchItems,G9,containment_family,299.802023,G3,300.348337,0.001822
23,sf_1,hot,QG3,QG3_RecommendationByGenreAndSubtype,G8,containment_family,10.417329,G0,10.727311,0.029756
28,sf_1,hot,QG8,QG8_AddPersonRoleToWatchItem,G3,root_local_family,0.373201,G0,0.375635,0.006522
29,sf_1,hot,QG9,QG9_TopRatedSeriesByGenre,G7,containment_family,4.078044,G2,159.413392,38.090653


,scale_label,run_phase,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret,activated_regret
0,sf0_25,hot,QG1,QG1_WatchItemById,G9,containment_family,0.205884,G0,0.285956,0.388919,0.109784
2,sf0_25,hot,QG2,QG2_WatchItemByTitle,G9,containment_family,0.282835,G0,0.297968,0.053505,0.042110
6,sf0_25,hot,QG6,QG6_EpisodesOfSeries,G2,root_local_family,0.632236,G7,0.918998,0.453568,0.453568
7,sf0_25,hot,QG7,QG7_UpdateWatchItemMetadata,G8,containment_family,0.262327,G0,0.396513,0.511524,0.021274
10,sf0_50,hot,QG1,QG1_WatchItemById,G7,containment_family,0.192424,G0,0.270370,0.405073,0.311679
12,sf0_50,hot,QG2,QG2_WatchItemByTitle,G7,containment_family,0.285235,G0,0.306578,0.074828,0.074828
17,sf0_50,hot,QG7,QG7_UpdateWatchItemMetadata,G7,containment_family,0.218918,G0,0.384026,0.754197,0.063108
18,sf0_50,hot,QG8,QG8_AddPersonRoleToWatchItem,G7,containment_family,0.358465,G0,0.401465,0.119956,0.049870
20,sf_1,hot,QG1,QG1_WatchItemById,G9,containment_family,0.213258,G0,0.271723,0.274153,0.004347
22,sf_1,hot,QG2,QG2_WatchItemByTitle,G8,containment_family,0.309479,G0,0.316557,0.022872,0.022872


,scale_label,near_best_preservation,mean_activated_regret
0,sf0_25,0.8,0.062674
1,sf0_50,0.7,0.049949
2,sf_1,1.0,0.003540


Saved to: /home/jovyan/privado/framework evaluation approachs/framework with dataset imdb oficial/results/cross_scale_analysis_final
